---
title: "Part 11: Polars & the Expression API"
---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sambaiga/ds-mlops-path/blob/main/tutorials/01-python-basics/11-polars.ipynb) [![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue.svg?logo=jupyter&logoColor=white)](https://raw.githubusercontent.com/sambaiga/ds-mlops-path/main/tutorials/01-python-basics/11-polars.ipynb)

Your pandas pipeline handles 10,000 rows without complaint. Then the dataset grows to 10 million rows and a `groupby` that used to finish instantly now takes 40 seconds. Polars is the answer: a DataFrame library written in Rust that uses lazy evaluation to optimize queries before reading a single byte -- and parallel execution to make full use of every CPU core.

Part 10 (`10-combining-reshaping.ipynb`) built the multi-table foundation you'll translate here. The same two datasets are used so every operation can be compared directly. Polars concepts carry forward into the MLOps section and streaming pipelines in Part 7.

> Callout markers used throughout this notebook are explained on the [book cover page](../../index.qmd#callout-guide).

In [ ]:
import numpy as np
import polars as pl

# Polars reads CSV directly; no separate parse step needed for standard columns
df = pl.read_csv("data/university_analytics.csv")
print(f"shape : {df.shape}")
df.head()

## When pandas Is Not Enough

Your pandas pipeline works perfectly at 50,000 rows. The `groupby` runs in a second, the merge finishes before you blink, and everything fits in memory with room to spare. Then someone drops a 10-million-row export on your desk. The same pipeline now takes three minutes, swaps to disk halfway through, and your laptop fan starts spinning like it's trying to take off.

This isn't a pandas bug. It is a design trade-off. pandas processes one column at a time on a single CPU thread, using a Python object model that was designed for flexibility rather than throughput. For most interactive data science that trade-off is exactly right. But when data gets large enough to feel slow, you need a different tool.

**Polars** ([pola.rs](https://pola.rs)) was written from scratch in Rust by Ritchie Vink in 2020. It stores data in Apache Arrow format, executes operations across all CPU cores in parallel, and evaluates expressions *lazily*: building a query plan and optimising it before touching a single row. The same operations that take minutes in pandas can take seconds in Polars, without changing what the code looks like.

### How it compares

| | pandas | Polars |
| --- | --- | --- |
| Core language | Python / C | Rust |
| Execution | Single-threaded | Multi-threaded |
| Memory model | NumPy / Python objects | Apache Arrow |
| Evaluation | Eager (runs immediately) | Lazy by default (`scan_*` → `.collect()`) |
| API style | Index-based, many methods | Expression-based, composable |
| Ecosystem maturity | Very mature (2008) | Fast-growing (2020) |
| When to use | Up to ~5M rows; default choice | Large files, performance-critical pipelines |

You don't have to choose one forever. Many production pipelines read and clean with Polars for speed, then convert to pandas for the parts of the ML ecosystem that expect it.

### Already in your environment

```bash
uv add polars          # for a standalone project
```

Official docs and user guide: [docs.pola.rs](https://docs.pola.rs/)

::: {.callout-note collapse="true" icon=false}
## Learning Objectives

By the end of Part 11 (Polars) you will be able to:

| # | Skill | Covered in |
|---|---|---|
| 1 | Build and inspect Polars DataFrames and compare them to pandas | Sec. 1 |
| 2 | Use `pl.col()` expressions to select, filter, and derive columns | Sec. 2 |
| 3 | Aggregate data with `group_by` and the expression API | Sec. 3 |
| 4 | Understand eager vs lazy evaluation and use `LazyFrame` | Sec. 4 |
| 5 | Parse and manipulate dates with Polars's built-in datetime support | Sec. 5 |
| 6 | Translate common pandas operations to Polars equivalents | Sec. 6 |
| 7 | Compute per-group window stats with `over()` and process large files with `scan_csv` | Sec. 7 |
:::


## 1. DataFrame Construction

Building a Polars DataFrame from a Python dict looks almost identical to pandas. The difference shows up in the schema: Polars always knows the dtype of every column, and it tells you explicitly:

In [ ]:
sample = pl.DataFrame(
    {
        "student_id": ["s001", "s002", "s003"],
        "midterm_score": [0.62, 0.78, 0.91],
        "gender": ["F", "M", "F"],
    }
)
print(sample.schema)
sample

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: Polars enforces strict dtypes; pandas 3 infers them</span><br><br>
Every Polars column has a fixed dtype from creation: <code>pl.Utf8</code> for strings, <code>pl.Float64</code> for floats, <code>pl.Int64</code> for integers. Polars raises an error if you try to mix types within a column. Pandas 3's <code>str</code> dtype is conceptually similar but more permissive: it silently allows <code>None</code> values in a string column. Polars is stricter, which makes bugs surface earlier.
</div>

The first checks after loading data are the same in Polars as in pandas, but the method names differ slightly:

In [ ]:
print(f"shape      : {df.shape}")
print(f"dtypes     :\n{df.dtypes}")
df.describe()

<div style='background:#F5F3FF;border-left:5px solid #7C3AED;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#5B21B6;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: Use <code>df.schema</code> to see column names and dtypes in one dict-like view</span><br><br>
<code>df.schema</code> returns a <code>Schema</code> object, a mapping from column name to Polars dtype. It is the fastest way to confirm that <code>read_csv</code> inferred the right types before any analysis starts.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 1 - First Look at the Polars DataFrame</span><br><br>
<b>Goal:</b> Print the schema of <code>df</code>, then use <code>df.describe()</code> to get summary statistics. Compare the dtype labels to the pandas output from Part 8.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>print(df.schema)
df.describe()</pre>
</div>

In [ ]:
# TODO: print schema, then call df.describe()
...

## 2. The Expression API: `select`, `filter`, `with_columns`

The biggest difference from pandas is that Polars doesn't use bracket indexing for column selection or boolean filtering. Instead, every operation goes through `pl.col()` expressions, which describe what to compute without computing it immediately.

In [ ]:
# Select specific columns -- equivalent to df[["col_a", "col_b"]] in pandas
df.select(["student_id", "midterm_score", "final_score"]).head(3)

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: <code>pl.col()</code> is an expression, not a value</span><br><br>
<code>pl.col("midterm_score")</code> describes a column reference, not the column itself. Expressions compose: <code>pl.col("a") + pl.col("b")</code> describes an addition without performing it. Polars evaluates all expressions in a <code>.select()</code> or <code>.with_columns()</code> call together, which is what lets it parallelize across columns automatically.
</div>

`filter` is the Polars equivalent of boolean indexing:

In [ ]:
# filter -- equivalent to df[df["midterm_score"] > 90] in pandas
high_scorers = df.filter(pl.col("midterm_score") > 90)
print(f"high scorers: {len(high_scorers)} of {len(df)} students")
high_scorers.head()

Combine conditions with `&` and `|`, same as pandas, but directly inside `filter` with `pl.col()` expressions:

In [ ]:
# combine conditions: failed AND no internet access
failed_no_internet = df.filter(
    (pl.col("final_grade") == "F") & (pl.col("has_internet") == False)  # noqa: E712
)
print(f"failed with no internet: {len(failed_no_internet)}")

`with_columns` adds or replaces columns. The result is always a new DataFrame; the original is never modified:

In [ ]:
# Add average_marks -- equivalent to df["average_marks"] = ... in pandas
df = df.with_columns(average_marks=((pl.col("midterm_score") + pl.col("final_score") + pl.col("project_score")) / 3))
df.select(["student_id", "average_marks"]).head()

<div style='background:#FEF2F2;border-left:5px solid #DC2626;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#991B1B;font-weight:bold'><i class="bi bi-bug-fill"></i> Common Mistake: Using bracket assignment instead of <code>with_columns</code></span><br><br>
<code>df["average_marks"] = ...</code> raises an error in Polars, unlike pandas. Polars DataFrames are immutable by design: the only way to add or modify a column is to call <code>.with_columns()</code> and reassign the result. This enforces the copy-on-write discipline explicitly, rather than relying on the runtime to enforce it as pandas 3 does.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 2 - Filter and Derive</span><br><br>
<b>Goal:</b> Add a <code>passed</code> column that is <code>True</code> when <code>average_marks >= 0.5</code>, using <code>.with_columns()</code> and <code>pl.col("average_marks") >= 0.5</code>. Then filter to passing students and print how many there are.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>df = df.with_columns(
    passed=(pl.col("average_marks") >= 0.5)
)
df.filter(pl.col("passed")).shape[0]</pre>
</div>

In [ ]:
# TODO: add "passed" column, filter to passing students, print count
...

<div style='background:#F5F3FF;border-left:5px solid #7C3AED;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#5B21B6;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: Use <code>polars.selectors</code> to select columns by type, not by name</span><br><br>
<code>polars.selectors</code> (aliased as <code>cs</code>) provides type-based column selectors, so you can write expressions like "all numeric columns" or "all string columns" without listing names:

<pre style='background:#F4F5F6;padding:10px;border-radius:4px;font-size:0.9em'>import polars.selectors as cs

# Normalise every numeric column in one expression
df.with_columns(
    (cs.numeric() - cs.numeric().mean()) / cs.numeric().std()
)

# Select only string columns
df.select(cs.string())</pre>

Common selectors: <code>cs.numeric()</code>, <code>cs.string()</code>, <code>cs.boolean()</code>, <code>cs.temporal()</code>, <code>cs.by_dtype(pl.Float64)</code>. Combine them with <code>|</code> (union) or <code>&amp;</code> (intersection): <code>cs.numeric() | cs.boolean()</code>. This is especially useful in feature engineering pipelines where the set of columns may change between runs.
</div>

## 3. GroupBy and Aggregation

Polars `group_by` uses the same split-apply-combine concept as pandas, but the aggregation is specified as a list of expressions inside `.agg()`, not as a method chained after `.mean()` or `.sum()`:

In [ ]:
df.group_by("program").agg(
    pl.col("average_marks").mean().alias("mean_marks"),
    pl.col("average_marks").std().alias("std_marks"),
    pl.len().alias("count"),
).sort("program")

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: <code>.agg()</code> takes a list of expressions, each producing one output column</span><br><br>
In pandas, <code>groupby("program")["average_marks"].agg(["mean", "std", "count"])</code> returns a DataFrame with the stats as columns. In Polars the same result comes from passing named expressions to <code>.agg()</code>: <code>pl.col("average_marks").mean().alias("mean_marks")</code>. The explicit <code>.alias()</code> is required when the default auto-generated name would be ambiguous.
</div>

Grouping by more than one column works the same way:

In [ ]:
df.group_by(["program", "gender"]).agg(
    pl.col("average_marks").mean().alias("mean_marks"),
).sort(["program", "gender"])

<div style='background:#F5F3FF;border-left:5px solid #7C3AED;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#5B21B6;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: Sort the result explicitly: <code>group_by</code> order is not guaranteed</span><br><br>
Polars <code>group_by</code> doesn't preserve or sort the output by the grouping key. Always chain <code>.sort("key_col")</code> after <code>.agg()</code> if the output order matters. The pandas equivalent sorted by default in older versions but no longer guarantees it either.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 3 - Pass Rate by Program</span><br><br>
<b>Goal:</b> Group <code>df</code> by <code>program</code> and compute, for each group, the fraction of students whose <code>continue_drop</code> is <code>False</code>. In Polars, <code>(pl.col("continue_drop") == False).mean()</code> gives the fraction directly.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>df.group_by("program").agg(
    drop_fraction=(pl.col("continue_drop") == False).mean()
).sort("program")</pre>
</div>

In [ ]:
# TODO: group by program, compute pass fraction per program, sort by program
...

When a vectorised expression (`pl.col("x") * 2`) can't express your logic, `map_elements()` applies a Python function element-wise. This is slower than expressions but necessary for custom string parsing, API calls, or complex conditionals that Polars can't optimise.

In [ ]:
def grade_label(score: float) -> str:
    if score >= 85:
        return "A"
    elif score >= 70:
        return "B"
    elif score >= 55:
        return "C"
    return "F"


graded = df.with_columns(pl.col("midterm_score").map_elements(grade_label, return_dtype=pl.String).alias("grade"))
graded.select(["student_id", "midterm_score", "grade"]).head(6)

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: Always pass <code>return_dtype</code> to <code>map_elements()</code></span><br><br>
<code>map_elements()</code> requires <code>return_dtype</code> to tell Polars the output type. Without it, Polars infers the type from the first element which can silently mismatch later rows. Always specify <code>return_dtype=pl.String</code>, <code>pl.Int64</code>, etc.
</div>

<div style='background:#FEF2F2;border-left:5px solid #DC2626;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#991B1B;font-weight:bold'><i class="bi bi-bug-fill"></i> Common Mistake: Using the removed <code>.apply()</code> method</span><br><br>
<code>apply()</code> was the pre-1.0 name for this function and is now removed. Any notebook or tutorial using <code>.apply()</code> on a Polars Series will fail on Polars 1.x. The fix is always a one-word rename to <code>.map_elements()</code> plus adding <code>return_dtype</code>.
</div>

## 4. LazyFrame and Lazy Evaluation

Every Polars operation above ran eagerly: it computed the result immediately and returned a new DataFrame. `df.lazy()` switches to lazy mode. In lazy mode, operations build a *query plan* instead of executing immediately. `.collect()` runs the optimized plan.

![Polars lazy evaluation: scan_csv, filter, and group_by each add a step to the query plan without reading data. Calling .collect() triggers the query optimizer, which applies predicate pushdown and projection pruning before executing a single parallel read.](figs/polars-lazy-evaluation.svg){fig-alt="Five boxes connected by arrows: scan_csv (purple dashed), filter (purple dashed), group_by (purple dashed), Query Optimizer (blue), DataFrame result (green). .collect() label on the arrow into the optimizer. Two annotation boxes below explain why lazy evaluation matters and the eager vs lazy distinction."}

In [ ]:
# Build a query plan -- nothing executes yet
lazy_query = (
    df.lazy()
    .filter(pl.col("has_internet") == 1)
    .group_by("program")
    .agg(pl.col("average_marks").mean().alias("mean_marks"))
    .sort("mean_marks", descending=True)
)

# Inspect the query plan before running
print(lazy_query.explain())

In [ ]:
# Execute the plan and get the result
lazy_query.collect()

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: Lazy evaluation lets Polars optimize the query before running it</span><br><br>
When you call <code>.collect()</code>, Polars analyses the whole query plan: it pushes filters as early as possible (predicate pushdown), drops columns that are not needed (projection pushdown), and parallelizes independent operations. The same query in eager mode runs each step sequentially with no cross-step optimization. For large datasets the difference is significant; for small ones it's negligible. The API is identical apart from the <code>.lazy()</code> / <code>.collect()</code> bookends.
</div>

<div style='background:#FEF2F2;border-left:5px solid #DC2626;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#991B1B;font-weight:bold'><i class="bi bi-bug-fill"></i> Common Mistake: Forgetting <code>.collect()</code> and treating a <code>LazyFrame</code> as a result</span><br><br>
<code>df.lazy().filter(...).group_by(...).agg(...)</code> returns a <code>LazyFrame</code>, not a <code>DataFrame</code>. Printing it shows the query plan, not the data. Call <code>.collect()</code> to get a real <code>DataFrame</code>. If you need the shape, columns, or to iterate over rows, you must collect first.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 4 - Lazy Top Schools</span><br><br>
<b>Goal:</b> Rewrite the top-schools capstone from Part 8 using a lazy query: filter to students with <code>internet == 1</code>, group by <code>school_id</code>, take the mean <code>average_marks</code>, sort descending, and <code>.collect()</code> the top 5.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>top_schools = (
    df.lazy()
    .filter(pl.col("has_internet") == 1)
    .group_by("school_id")
    .agg(pl.col("average_marks").mean().alias("mean_marks"))
    .sort("mean_marks", descending=True)
    .limit(5)
    .collect()
)
top_schools</pre>
</div>

In [ ]:
# TODO: lazy top-schools query
...

## 4.5 SQL Interface

Polars 1.x ships a built-in SQL interface. `pl.SQLContext` registers one or more DataFrames under names and lets you query them with standard SQL. This is useful for teams comfortable with SQL, for migrating existing SQL queries to Polars, and for ad-hoc exploration without learning the full expression API:

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: SQL queries run on lazy DataFrames and benefit from the same query optimization</span><br><br>
<code>pl.SQLContext</code> compiles SQL into Polars' lazy query plan. The optimizer applies the same predicate pushdown and projection pushdown as a hand-written <code>.lazy().filter().select()</code> chain. You get SQL's readability and Polars' performance for free.
</div>

In [ ]:
# Register DataFrames under SQL table names
ctx = pl.SQLContext(students=df, eager=True)

# Query 1: top-performing programs: pure SQL
result = ctx.execute("""
    SELECT
        program,
        AVG(final_score)    AS mean_score,
        COUNT(*)            AS n_students
    FROM students
    GROUP BY program
    ORDER BY mean_score DESC
""")
print(result)

# Query 2: filter + select: same as df.filter().select() but SQL syntax
high_scorers_sql = ctx.execute("""
    SELECT student_id, final_score
    FROM students
    WHERE final_score > 0.9
    ORDER BY final_score DESC
    LIMIT 5
""")
print(high_scorers_sql)

## 5. Date/Time in Polars

Polars can parse dates directly at read time with `try_parse_dates=True`. There is no separate `to_datetime` step for standard ISO-format date strings:

In [ ]:
attendance = pl.read_csv("data/daily_attendance.csv", try_parse_dates=True)
print(attendance.dtypes)
attendance.head()

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: Polars parses dates at load time; pandas requires an explicit conversion step</span><br><br>
In pandas, date columns read as <code>str</code> dtype until you call <code>pd.to_datetime()</code>. Polars with <code>try_parse_dates=True</code> infers and parses ISO-format dates automatically during <code>read_csv</code>, producing a <code>pl.Date</code> or <code>pl.Datetime</code> column. This means filtering and arithmetic work on dates immediately, without a conversion step.
</div>

Date-component extraction uses the `.dt` namespace, similar to pandas:

In [ ]:
# Extract year, month, and weekday from the date column
attendance.select(
    [
        "date",
        pl.col("date").dt.year().alias("year"),
        pl.col("date").dt.month().alias("month"),
        pl.col("date").dt.weekday().alias("weekday"),  # Mon=1 ... Sun=7
    ]
).head()

Resampling to weekly or monthly means uses `group_by_dynamic` instead of pandas' `resample`:

In [ ]:
# Monthly mean attendance rate per school using group_by_dynamic
school_300 = attendance.filter(pl.col("school_id") == 300)

school_300.sort("date").group_by_dynamic("date", every="1mo", group_by="school_id").agg(
    pl.col("attendance_rate").mean().alias("monthly_mean")
)

<div style='background:#F5F3FF;border-left:5px solid #7C3AED;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#5B21B6;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: <code>group_by_dynamic</code> replaces <code>resample</code> in Polars</span><br><br>
<code>series.resample("M").mean()</code> in pandas becomes <code>df.sort("date").group_by_dynamic("date", every="1mo").agg(...)</code> in Polars. The <code>sort</code> before <code>group_by_dynamic</code> is required: Polars raises an error if the time column is not sorted.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 5 - Monthly Attendance by School</span><br><br>
<b>Goal:</b> Use <code>group_by_dynamic</code> to compute the mean <code>attendance_rate</code> per school per month across all schools in the <code>attendance</code> table. Sort by <code>date</code> before grouping.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>attendance.sort("date").group_by_dynamic(
    "date", every="1mo", group_by="school_id"
).agg(
    pl.col("attendance_rate").mean().alias("monthly_mean")
).sort(["school_id", "date"])</pre>
</div>

In [ ]:
# TODO: monthly mean attendance per school using group_by_dynamic
...

## 6. Migration Patterns: pandas to Polars

The table below maps the most common pandas operations from Parts 8-10 to their Polars equivalents. The concepts are the same; the API surface is different:

| Task | pandas | Polars |
|---|---|---|
| Load CSV | `pd.read_csv(path)` | `pl.read_csv(path, try_parse_dates=True)` |
| Select columns | `df[["a", "b"]]` | `df.select(["a", "b"])` |
| Filter rows | `df[df["col"] > 0.5]` | `df.filter(pl.col("col") > 0.5)` |
| Add column | `df["c"] = df["a"] + df["b"]` | `df.with_columns(c=pl.col("a") + pl.col("b"))` |
| Rename column | `df.rename(columns={"old": "new"})` | `df.rename({"old": "new"})` |
| GroupBy + agg | `df.groupby("k")["v"].mean()` | `df.group_by("k").agg(pl.col("v").mean())` |
| Sort | `df.sort_values("col", ascending=False)` | `df.sort("col", descending=True)` |
| Missing values | `df.isna().sum()` | `df.null_count()` |
| Fill missing | `df["col"].fillna(val)` | `df.with_columns(pl.col("col").fill_null(val))` |
| One-hot encode | `pd.get_dummies(df["col"])` | `df.to_dummies("col")` |
| Value counts | `df["col"].value_counts()` | `df["col"].value_counts()` |
| Lazy query | _(eager only)_ | `df.lazy()` ... `.collect()` |
| Resample | `series.resample("M").mean()` | `df.sort("date").group_by_dynamic("date", every="1mo").agg(...)` |

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: Polars is strict; pandas is permissive</span><br><br>
Polars raises errors early: wrong dtype in a column, missing <code>.collect()</code> on a LazyFrame, unsorted data before <code>group_by_dynamic</code>. Pandas absorbs most of these quietly, sometimes silently producing a wrong result. The Polars errors feel strict at first but save time in larger pipelines where a wrong-but-valid pandas result would surface much later.
</div>

## 7. Window Functions with `over()` and Streaming Large Files

Two Polars strengths that have no direct pandas equivalent: `over()` for grouped window calculations without reducing rows, and `scan_csv`/`scan_parquet` for processing files that don't fit in memory.

### 7a. The `over()` Expression

In pandas, adding a per-group statistic back to the original rows requires `groupby().transform()`. In Polars, `pl.col("x").mean().over("group")` does the same thing inside a single `with_columns()` call. It computes the aggregation per group and broadcasts the result back to every row, aligned to the original index automatically.

In [ ]:
# Add the per-program mean and std to every student row, in one expression
df = df.with_columns(
    program_mean=pl.col("average_marks").mean().over("program"),
    program_std=pl.col("average_marks").std().over("program"),
)

# Z-score: how many standard deviations from the program mean?
df = df.with_columns(marks_zscore=(pl.col("average_marks") - pl.col("program_mean")) / pl.col("program_std"))

df.select(["student_id", "program", "average_marks", "program_mean", "marks_zscore"]).head(6)

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: <code>over()</code> computes group stats without reducing rows</span><br><br>
<code>group_by().agg()</code> produces one row per group: the result has as many rows as there are unique group values. <code>.over("group")</code> inside <code>with_columns()</code> keeps all original rows and attaches the group stat to each one. This is the right tool for feature engineering: every student in the <code>"Engineering"</code> program gets the engineering mean attached to their row, ready for modelling without a join.
</div>

In [ ]:
# Rolling mean per school using over() -- group-aware rolling window
attendance = pl.read_csv("data/daily_attendance.csv", try_parse_dates=True)

attendance_with_rolling = attendance.sort(["school_id", "date"]).with_columns(
    rolling_avg=pl.col("attendance_rate").rolling_mean(window_size=5, min_periods=3).over("school_id")
)

attendance_with_rolling.filter(pl.col("school_id") == 300).head(8)

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 6 - Per-Program Z-Score</span><br><br>

<b>Goal:</b> Using <code>over()</code>, add a column <code>gender_mean_marks</code> that is the mean <code>average_marks</code> within each gender group. Then add <code>gender_zscore = (average_marks - gender_mean_marks) / gender_std_marks</code>. Print the top 3 students by z-score in each gender.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>df_z = df.with_columns(
    gender_mean = pl.col("average_marks").mean().over("gender"),
    gender_std  = pl.col("average_marks").std().over("gender"),
).with_columns(
    gender_zscore = (pl.col("average_marks") - pl.col("gender_mean")) / pl.col("gender_std")
)
df_z.sort("gender_zscore", descending=True).select(["student_id", "gender", "average_marks", "gender_zscore"]).head(6)</pre>
</div>

In [ ]:
# TODO: per-gender z-score using over()
...

### 7b. Streaming: Processing Files Larger Than Memory

`pl.scan_csv()` and `pl.scan_parquet()` read a file lazily: no data loads until `.collect()` is called. Combined with Polars' streaming engine, filters and projections execute in chunks, so the working memory never exceeds what the query actually needs, even if the source file is larger than RAM.

The API is identical to eager reads; the difference is `scan_` instead of `read_`.

In [ ]:
# scan_csv: no data loads here, only a query plan is built
lazy_scan = (
    pl.scan_csv("data/university_analytics.csv")
    .filter(pl.col("final_score") > 80)
    .select(["student_id", "program", "final_score"])
    .sort("final_score", descending=True)
)

# .collect() executes with streaming (chunks through the file)
result = lazy_scan.collect(streaming=True)
print(f"Rows matching filter: {len(result)}")
result.head()

<div style='background:#F5F3FF;border-left:5px solid #7C3AED;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#5B21B6;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: Use Parquet instead of CSV for large files</span><br><br>
CSV is plain text: Polars must read every byte to find the columns you asked for. Parquet is columnar: <code>scan_parquet()</code> reads only the columns referenced in your query, skipping all others. A file with 50 columns where you only need 3 loads roughly 3/50th of the data. For production pipelines with large files, converting a CSV to Parquet once and scanning Parquet from then on is a common and effective optimisation.
<pre style='background:#F4F5F6;padding:10px;border-radius:4px;font-size:0.9em'># Convert once
df.write_parquet("data/university_analytics.parquet")

# All future reads: columnar, faster, smaller
pl.scan_parquet("data/university_analytics.parquet")   .select(["student_id", "final_score"])   .filter(pl.col("final_score") > 80)   .collect(streaming=True)</pre>
</div>

<div style='background:#FEF2F2;border-left:5px solid #DC2626;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#991B1B;font-weight:bold'><i class="bi bi-bug-fill"></i> Common Mistake: Not sorting before a rolling window in Polars</span><br><br>
<code>pl.col("rate").rolling_mean(5).over("school_id")</code> operates on whatever order the rows are in. Without <code>.sort(["school_id", "date"])</code> first, the 5-row window picks up rows in an arbitrary order and produces meaningless values. Polars doesn't sort automatically; it raises an error only for <code>group_by_dynamic</code>. For <code>rolling_mean().over()</code>, the wrong-order mistake is silent.
</div>

## Capstone: Reproduce the pandas Capstone Exercises in Polars

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Capstone Exercise - Top Schools, Risk Report, and Regional Summary</span><br><br>
<b>Goal:</b> Reproduce three capstones from earlier parts using Polars:
<ol>
<li><b>Top schools (Part 8):</b> Filter to students with <code>internet == 1</code>, group by <code>school_id</code>, compute mean <code>average_marks</code>, sort descending, show top 5.</li>
<li><b>Risk report (Part 9):</b> Add a <code>grade</code> column based on <code>average_marks</code> using <code>pl.when().then().otherwise()</code>. Print grade distribution as proportions.</li>
<li><b>Regional summary (Part 10):</b> Create a <code>schools</code> DataFrame mapping <code>school_id</code> to <code>region</code>, join it onto <code>df</code>, and compute mean marks per region.</li>
</ol>
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'># 1. Top schools
top_schools = (
    df.lazy()
    .filter(pl.col("has_internet") == 1)
    .group_by("school_id")
    .agg(pl.col("average_marks").mean().alias("mean_marks"))
    .sort("mean_marks", descending=True)
    .limit(5)
    .collect()
)

# 2. Risk report -- pl.when().then().otherwise() replaces df.apply(letter_grade)
df_graded = df.with_columns(
    grade=pl.when(pl.col("average_marks") >= 0.8).then(pl.lit("A"))
           .when(pl.col("average_marks") >= 0.6).then(pl.lit("B"))
           .when(pl.col("average_marks") >= 0.4).then(pl.lit("C"))
           .otherwise(pl.lit("D"))
)
grade_distribution = df_graded["grade"].value_counts(normalize=True).sort("grade")

# 3. Regional summary
import polars as pl
school_ids = df["school_id"].unique().sort()
regions = ["North", "South", "East", "West"]
schools = pl.DataFrame({
    "school_id": school_ids,
    "region": [regions[i % 4] for i in range(len(school_ids))],
})
merged = df.join(schools, on="school_id", how="left")
regional_summary = merged.group_by("region").agg(
    pl.col("average_marks").mean().alias("mean_marks")
).sort("region")</pre>
</div>

In [ ]:
# TODO: reproduce top schools, risk report, and regional summary in Polars
...

## Further Reading

| Resource | Why it matters |
|---|---|
| [Polars documentation: Getting started](https://docs.pola.rs) | The primary reference; the user guide covers lazy evaluation, the expression API, and performance tips |
| [Polars migration guide from pandas](https://docs.pola.rs/user-guide/migration/pandas/) | A direct comparison of common pandas idioms rewritten in Polars; read this alongside this notebook |
| Vink, R. (2023). [Modern Polars](https://kevinheavey.github.io/modern-polars/) | Free online book that mirrors the official pandas docs structure, useful for side-by-side comparison |
| [Polars expression cheat sheet](https://docs.pola.rs/user-guide/expressions/) | `over()`, `rolling_*`, selectors, and string ops in one place |
| Ritchie Vink, creator of Polars, explains the design in the [Polars blog](https://pola.rs/posts/) | The posts on the expression API and lazy evaluation explain why certain pandas patterns don't translate |

## Summary

| Concept | Key rule |
|---|---|
| `pl.read_csv(try_parse_dates=True)` | Load data with automatic date inference |
| `df.schema` | Dict-like view of column names and Polars dtypes |
| `df.select([...])` | Select columns; equivalent to `df[["a", "b"]]` in pandas |
| `pl.col("name")` | Column expression; compose with arithmetic, comparisons, and string ops |
| `df.filter(expr)` | Keep rows where the expression is True |
| `df.with_columns(alias=expr)` | Add or overwrite columns; returns a new DataFrame |
| `df.group_by(key).agg([exprs])` | Split-apply-combine with explicit expression list |
| `.alias("name")` | Rename an expression's output column |
| `df.lazy()` ... `.collect()` | Defer execution so Polars can optimize the query plan |
| `pl.when().then().otherwise()` | Conditional column derivation; replaces `apply(lambda ...)` |
| `df.join(other, on=key, how=...)` | SQL-style join; same `how` values as pandas `merge` |
| `.dt.year()`, `.dt.month()` | Date-component extraction in the `.dt` namespace |
| `group_by_dynamic("date", every="1mo")` | Time-window grouping; requires sorted input |
| `df.null_count()` | Count missing values per column; replaces `df.isna().sum()` |
| `.over("group")` | Broadcast a group aggregation back to every row; no join needed |
| `pl.col("x").rolling_mean(n).over("g")` | Per-group rolling window; sort by group and time first |
| `pl.scan_csv()` / `pl.scan_parquet()` | Lazy file scan; push-down filters execute without loading full file |
| `.collect(streaming=True)` | Execute in chunks; keeps peak memory low for files larger than RAM |